### Import và cấu hình session

In [2]:

import os
import re
import csv
import time
import random
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlsplit, urlunsplit, parse_qsl, urlencode

s = requests.Session()

### Các hàm tiện ích

In [2]:
def get_soup(url, timeout=20):
    """
    Gửi request GET và trả về BeautifulSoup nếu thành công.
    Trả về None nếu status_code != 200.
    """
    r = s.get(url, timeout=timeout)
    if r.status_code != 200:
        return None
    return BeautifulSoup(r.text, "html.parser")


def clean_text(t):
    """
    Chuẩn hoá chuỗi:
    - Nếu None -> ""
    - Gộp nhiều khoảng trắng thành 1
    - strip() 2 đầu
    """
    return re.sub(r"\s+", " ", (t or "")).strip()


def parse_price_ty_vnd(text):
    """
    Nhận vào text giá (ví dụ: '3 tỷ 200 triệu' hoặc '4 tỷ') và
    trả về:
      - total: giá quy đổi sang đơn vị TỶ VND (float)
      - raw:  chuỗi gốc đã lower/clean
    """
    if not text:
        return None, None
    t = clean_text(text).lower()
    raw = t
    words = t.split(" ")

    total = 0.0
    for i in range(len(words)):
        if "tỷ" in words[i] and i > 0:
            total += int(words[i - 1])
        elif "triệu" in words[i] and i > 0:
            total += int(words[i - 1]) / 1000.0
    return total, raw


def parse_area_m2(text):
    """
    Parse diện tích m2 từ chuỗi, hỗ trợ các dạng:
    - '78 m2'
    - '78 m 2'
    - '78 m^2'
    - '78 m²'
    Trả về float số m2 hoặc None nếu không parse được.
    """
    if not text:
        return None
    t = clean_text(text).lower()

    # Chuẩn hoá các biến thể 'm 2', 'm ^ 2', 'm²' -> 'm2'
    t = t.replace("m²", "m2")
    t = re.sub(r"m\s*\^\s*2", "m2", t)  # m ^ 2
    t = re.sub(r"m\s*2", "m2", t)       # m 2

    m = re.search(r"([\d\.,]+)\s*m2\b", t)
    if not m:
        return None

    s = m.group(1).replace(",", ".")
    s = re.sub(r"[^0-9.]", "", s)  # bỏ ký tự lạ (phòng trường hợp)

    try:
        return float(s)
    except ValueError:
        return None

def parse_so_tang_from_desc(text):
    """
    Tìm và ước lượng số tầng từ phần mô tả (giới thiệu).
    Một số mẫu hỗ trợ:
      - 'nhà 3 tầng'
      - '2 lầu'
      - '1 trệt 2 lầu'  -> 3
      - '1 trệt 1 lầu 1 sân thượng' -> 2 (tính trệt + lầu)
    Trả về int hoặc None nếu không tìm được.
    """
    if not text:
        return None

    t = clean_text(text).lower()

    floors = 0

    # Bắt các cụm: "3 tầng", "2 tang", "2 lầu", "2 lau"
    matches = re.findall(r"(\d+)\s*(tầng|tang|lầu|lau)", t)
    for num, _ in matches:
        floors += int(num)

    # Bắt các cụm: "1 trệt"
    tret_matches = re.findall(r"(\d+)\s*trệt", t)
    for num in tret_matches:
        floors += int(num)

    # Nếu không bắt được gì, trả về None
    if floors == 0:
        return None

    return floors

def first_int(text):
    """
    Trích số nguyên đầu tiên tìm được trong chuỗi (ví dụ '3 phòng ngủ' -> 3).
    Trả về None nếu không tìm thấy.
    """
    if not text:
        return None
    m = re.search(r"\d+", text)
    return int(m.group()) if m else None
    
def extract_info_attr_rows(soup):
    """
    Đọc từng dòng thuộc tính trong DOM Mogi dạng:
      <div class="info-attr clearfix">
        <span>Diện tích sử dụng</span>
        <span>78 m<sup>2</sup></span>
      </div>

    Trả về dict: { label_lower: value_text }
    """
    info = {}

    # Tất cả dòng thuộc tính
    rows = soup.select("div.info-attr.clearfix")

    # Fallback nếu DOM thay đổi đôi chút
    if not rows:
        container = soup.select_one("div.info-attrs.clearfix")
        if container:
            rows = container.select("div.info-attr")

    for row in rows:
        # Ưu tiên lấy 2 <span> con trực tiếp
        spans = row.find_all("span", recursive=False)
        if len(spans) >= 2:
            label = clean_text(spans[0].get_text(" ", strip=True)).lower()
            # get_text("", strip=True) để 'm' + <sup>2> -> 'm2'
            value = clean_text(spans[1].get_text("", strip=True))
            if label and value:
                info[label] = value
        else:
            # Trường hợp hiếm: label:value chung một thẻ
            t = clean_text(row.get_text(" ", strip=True))
            if ":" in t:
                k, v = t.split(":", 1)
                k = clean_text(k).lower()
                v = clean_text(v)
                if k and v:
                    info[k] = v

    return info


def pick_value(pairs, keys):
    """
    Từ dict pairs {label: value}, tìm giá trị đầu tiên mà label chứa
    bất kỳ từ khoá nào trong 'keys' (list các từ khoá, đã lower-case).
    """
    for lab, val in pairs.items():
        lab_l = lab.lower()
        if any(k in lab_l for k in keys):
            return val
    return None

### Parse trang chi tiết

In [3]:
def parse_detail_htmlparser(url):
    """
    Parse một trang chi tiết tin đăng trên Mogi:
    Trả về dict chỉ gồm các trường:
      - tieu_de
      - gia      (float, đơn vị: tỷ VND)
      - dia_chi
      - dien_tich_dat  (m2, float)
      - phong_ngu
      - phong_tam
      - so_tang
      - phap_ly
      - ngay_dang
    """
    soup = get_soup(url)
    if not soup:
        return None

    # Tiêu đề
    h1 = soup.find("h1")
    tieu_de = clean_text(h1.get_text(" ")) if h1 else None

    # Giá (từ div.price) -> parse về đơn vị TỶ VND
    price_el = soup.find("div", class_="price")
    gia_text = clean_text(price_el.get_text(" ")) if price_el else None
    gia_vnd, gia_raw = parse_price_ty_vnd(gia_text)
    # Ở đây ta dùng 'gia' là số float (tỷ VND)
    gia = gia_vnd

    # Địa chỉ
    addr_el = soup.find("div", class_="address")
    dia_chi = clean_text(addr_el.get_text(" ")) if addr_el else None

    # Giới thiệu (mô tả) – sẽ dùng để suy ra số tầng
    desc_el = soup.find("div", class_="info-content-body")
    gioi_thieu = clean_text(desc_el.get_text("\n")) if desc_el else None

    # Các thuộc tính trong info-attr
    pairs = extract_info_attr_rows(soup)
    # print(pairs)  # Nếu cần debug thì mở comment

    # Diện tích sử dụng / đất
    dt_sd_text  = pick_value(pairs, ["diện tích sử dụng"])
    dt_dat_text = pick_value(pairs, ["diện tích đất"])


    dien_tich_dat = parse_area_m2(dt_dat_text) if dt_dat_text else None
    # Nếu cần diện tích sử dụng thì có thể parse dt_sd_text tương tự

    # Phòng ngủ / phòng tắm
    phong_ngu_text = pick_value(pairs, ["phòng ngủ"])
    phong_tam_text = pick_value(pairs, ["nhà tắm"])

    phong_ngu = first_int(phong_ngu_text)
    phong_tam = first_int(phong_tam_text)

    # Ngày đăng
    ngay_dang = pick_value(pairs, ["ngày đăng"])

    # Pháp lý
    phap_ly = pick_value(pairs,["pháp lý"])

    # Số tầng: phân tích từ phần mô tả
    so_tang = parse_so_tang_from_desc(gioi_thieu)

    return {
        "tieu_de": tieu_de,
        "gia": gia,                    # float, đơn vị: tỷ VND
        "dia_chi": dia_chi,
        "dien_tich_dat": dien_tich_dat,
        "phong_ngu": phong_ngu,
        "phong_tam": phong_tam,
        "so_tang": so_tang,
        "phap_ly": phap_ly,
        "ngay_dang": ngay_dang
    }

### Trích xuất các link tin đăng từ trang listing

In [4]:

PATTERN_DETAIL = re.compile(r"-id(\d{5,})", re.I)


def extract_listing_links(listing_soup, base_url):
    """
    Từ một soup trang listing, trích tất cả link chi tiết hợp lệ của Mogi
    (có chứa '-idxxxxx').
    Trả về list URL đầy đủ (loại trùng).
    """
    links = []
    seen = set()

    if not listing_soup:
        return links

    for a in listing_soup.select("a[href]"):
        href = a.get("href", "")
        if not href or href.startswith("#") or "javascript:" in href:
            continue

        full = urljoin(base_url, href).split("?")[0]
        m = PATTERN_DETAIL.search(full)

        if ("mogi.vn" in full) and m and full not in seen:
            seen.add(full)
            links.append(full)

    return links


def set_cp_param(url, page: int) -> str:
    """
    Gắn (hoặc thay đổi) tham số cp=page trên URL:
      - Dùng cho phân trang: cp=1, cp=2, ...
    """
    parts = urlsplit(url)
    q = parse_qsl(parts.query, keep_blank_values=True)

    out = []
    seen_cp = False
    for k, v in q:
        if k.lower() == "cp":
            out.append(("cp", str(page)))
            seen_cp = True
        else:
            out.append((k, v))

    if not seen_cp:
        out.append(("cp", str(page)))

    new_query = urlencode(out, doseq=True)
    return urlunsplit((parts.scheme, parts.netloc, parts.path, new_query, parts.fragment))


def collect_links_by_cp(base_url,
                        start_page=1,
                        max_pages=30,
                        sleep_range=(1.0, 2.0),
                        break_no_new_pages=2):
    """
    Duyệt nhiều trang listing theo tham số cp (cp=1..N),
    thu thập tất cả link chi tiết mới (theo ad_id).
    Dừng lại nếu:
      - Không tải được trang, hoặc
      - 'break_no_new_pages' trang liên tiếp không có link mới.

    Trả về list tất cả các link chi tiết (URL).
    """
    all_links = []
    seen_ids = set()
    no_new = 0

    for page in range(start_page, start_page + max_pages):
        page_url = set_cp_param(base_url, page)
        soup = get_soup(page_url)
        if not soup:
            print("Dừng do không tải được trang:", page_url)
            break

        links = extract_listing_links(soup, page_url)

        # Lọc link mới theo ad_id
        new_links = []
        for u in links:
            m = PATTERN_DETAIL.search(u)
            adid = m.group(1) if m else None
            if adid and adid not in seen_ids:
                seen_ids.add(adid)
                new_links.append(u)

        print(f"Trang cp={page}: {len(new_links)}/{len(links)} link mới; tổng {len(all_links)+len(new_links)}")

        if not new_links:
            no_new += 1
        else:
            no_new = 0

        all_links.extend(new_links)

        # Nếu liên tiếp nhiều trang không có link mới -> dừng
        if no_new >= break_no_new_pages:
            print("Không thấy link mới trong nhiều trang liên tiếp → dừng.")
            break

        # Nghỉ ngẫu nhiên tránh bị chặn
        time.sleep(random.uniform(*sleep_range))

    return all_links

### Ghi dữ liệu và crawl chi tiết

In [5]:

def ensure_csv(file_path, fieldnames):
    """
    Mở file CSV ở chế độ append.
    Nếu file chưa tồn tại thì ghi header.
    Trả về (file_handle, csv_writer).
    """
    # Tạo thư mục nếu chưa tồn tại
    dir_name = os.path.dirname(file_path)
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)

    new = not os.path.exists(file_path)
    f = open(file_path, "a", newline="", encoding="utf-8-sig")
    w = csv.DictWriter(f, fieldnames=fieldnames)
    if new:
        w.writeheader()
    return f, w


def crawl_details_to_csv(detail_links,
                         out_csv="../data/extracted/mogi_raw.csv",
                         batch_size=100,
                         sleep_range=(1.0, 2.0)):
    """
    Nhận vào list link chi tiết (detail_links),
    parse từng link và ghi ra file CSV theo từng lô (batch_size).

    YÊU CẦU: parse_detail_htmlparser(url) phải trả về dict gồm các key:
      - tieu_de
      - gia
      - dia_chi
      - dien_tich_dat
      - phong_ngu
      - phong_tam
      - so_tang
      - phap_ly
      - ngay_dang
    """
    # Các cột sẽ ghi ra CSV (KHÔNG có ad_id)
    cols = [
        "tieu_de",
        "gia",
        "dia_chi",
        "dien_tich_dat",
        "phong_ngu",
        "phong_tam",
        "so_tang",
        "phap_ly",
        "ngay_dang"
    ]

    f, w = ensure_csv(out_csv, cols)
    written = 0
    batch = []

    try:
        for i, url in enumerate(detail_links, 1):
            try:
                item = parse_detail_htmlparser(url)
                if not item:
                    continue

                # Chỉ giữ các cột cần thiết
                row = {k: item.get(k) for k in cols}
                batch.append(row)

            except Exception as e:
                print("Lỗi parse:", url, e)

            # Ghi theo lô
            if len(batch) >= batch_size:
                w.writerows(batch)
                written += len(batch)
                print(f"Đã ghi {written} bản ghi vào {out_csv}")
                batch.clear()

            # Nghỉ tránh bị chặn
            time.sleep(random.uniform(*sleep_range))

        # Ghi nốt phần còn lại
        if batch:
            w.writerows(batch)
            written += len(batch)
            print(f"Đã ghi {written} bản ghi vào {out_csv}")

    finally:
        f.close()

### Cào dữ liệu

In [ ]:
base_url = "https://mogi.vn/ho-chi-minh/quan-thu-duc/mua-nha"

# 1. Thu thập link chi tiết từ nhiều trang listing
detail_links = collect_links_by_cp(
    base_url=base_url,
    start_page=1,
    max_pages=10,           # số trang muốn duyệt
    sleep_range=(1.0, 2.0), # thời gian nghỉ giữa các request listing
    break_no_new_pages=2
)

print("Tổng số link chi tiết thu được:", len(detail_links))

# 2. Crawl nội dung từng link chi tiết và ghi vào CSV
crawl_details_to_csv(
    detail_links=detail_links,
    out_csv="../data/extracted/mogi_dump.csv",
    batch_size=50,          # số bản ghi ghi 1 lần
    sleep_range=(1.0, 2.0)  # thời gian nghỉ giữa các request detail
)


Trang cp=1: 15/15 link mới; tổng 15
Trang cp=2: 15/15 link mới; tổng 30
Trang cp=3: 15/15 link mới; tổng 45
Trang cp=4: 15/15 link mới; tổng 60
Trang cp=5: 15/15 link mới; tổng 75
Trang cp=6: 15/15 link mới; tổng 90
Trang cp=7: 15/15 link mới; tổng 105
Trang cp=8: 15/15 link mới; tổng 120
Trang cp=9: 15/15 link mới; tổng 135
Trang cp=10: 15/15 link mới; tổng 150
Tổng số link chi tiết thu được: 150
Đã ghi 50 bản ghi vào ../data/extracted/mogi_dump.csv
Đã ghi 100 bản ghi vào ../data/extracted/mogi_dump.csv
Đã ghi 150 bản ghi vào ../data/extracted/mogi_dump.csv
